In [1]:
import os
import re
import time
import pandas as pd
from tqdm import tqdm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

# ==========================================
# 1. SETUP AMBIENTE E CARICAMENTO DATI
# ==========================================
# Sostituisci "input_dataset.csv" con il percorso reale del tuo file
INPUT_CSV = "./data/dataset_report_anonimizzati_llama_3_1_8B_4bit.csv"
OUTPUT_CSV_ANONIMO = "dataset_report_anonimizzati_Llama-3.1-8B_8bit_zero_shot_prompt_semplificato.csv"

if not os.path.exists(INPUT_CSV):
    raise FileNotFoundError(f"Il file '{INPUT_CSV}' non esiste.")

df_input = pd.read_csv(INPUT_CSV)

if "testo_originale" not in df_input.columns or "nome_file" not in df_input.columns:
    raise ValueError("Il DataFrame sorgente deve contenere le colonne 'testo_originale' e 'nome_file'.")

# Checkpoint ottimizzato: lettura singola pre-ciclo $O(1)$ lookup
file_processati = set()
if os.path.exists(OUTPUT_CSV_ANONIMO):
    df_check = pd.read_csv(OUTPUT_CSV_ANONIMO)
    if "nome_file" in df_check.columns:
        file_processati = set(df_check['nome_file'].values)

# ==========================================
# 2. ESTRAZIONE METADATI E PULIZIA TESTO
# ==========================================
def estrai_metadata(nome_file):
    base = nome_file.replace("_KeyResults.txt", "")
    periodo_match = re.search(r'([A-Za-z]{3}_\d{4})_-_([A-Za-z]{3}_\d{4})', base)

    if periodo_match:
        inizio_periodo = periodo_match.group(1).replace('_', ' ')
        fine_periodo = periodo_match.group(2).replace('_', ' ')
        periodo = f"{inizio_periodo} / {fine_periodo}"
    else:
        inizio_periodo = None
        fine_periodo = None
        periodo = base

    paese_match = re.match(r'^(.+?)_[A-Za-z]{3}_\d{4}', base)
    paese = paese_match.group(1).replace('_', ' ') if paese_match else base

    return {
        "paese": paese,
        "periodo": periodo,
        "inizio_periodo": inizio_periodo,
        "fine_periodo": fine_periodo,
        "nome_file": nome_file
    }
df_input = df_input.drop(columns="report_anonimo")
df_input

,nome_file,paese,periodo,inizio_periodo,fine_periodo,testo_originale
0,Gaza_Strip_Sep_2024_-_Apr_2025_KeyResults.txt,Gaza Strip,Sep 2024 / Apr 2025,Sep 2024,Apr 2025,"One year into the conflict, the risk of Famine..."
1,Afghanistan_Apr_2020_-_Nov_2020_KeyResults.txt,Afghanistan,Apr 2020 / Nov 2020,Apr 2020,Nov 2020,Food insecurity remains alarmingly high in Afg...
2,Afghanistan_Nov_2017_-_Feb_2018_KeyResults.txt,Afghanistan,Nov 2017 / Feb 2018,Nov 2017,Feb 2018,"During the 2017 post-harvest season, 33% of th..."
3,South_Sudan_Sep_2018_-_Mar_2019_KeyResults.txt,South Sudan,Sep 2018 / Mar 2019,Sep 2018,Mar 2019,"Based on the September IPC analysis, it is exp..."
4,Mozambique_Jun_2020_-_Sep_2020_KeyResults.txt,Mozambique,Jun 2020 / Sep 2020,Jun 2020,Sep 2020,The results of this Acute Food Insecurity pilo...
...,...,...,...,...,...,...
492,Haiti_Aug_2023_-_Jun_2024_KeyResults.txt,Haiti,Aug 2023 / Jun 2024,Aug 2023,Jun 2024,About 4.35 million people are experiencing hig...
493,Djibouti_Mar_2022_-_Dec_2022_KeyResults.txt,Djibouti,Mar 2022 / Dec 2022,Mar 2022,Dec 2022,For the current analysis period of March throu...
494,Djibouti_May_2013_-_May_2013_KeyResults.txt,Djibouti,May 2013 / May 2013,May 2013,May 2013,Food availability in the Republic of Djibouti ...
495,Madagascar_Sep_2024_-_Aug_2025_KeyResults.txt,Madagascar,Sep 2024 / Aug 2025,Sep 2024,Aug 2025,"Between September and December 2024, around 1...."


In [2]:


# ==========================================
# 3. DOWNLOAD GGUF E INIZIALIZZAZIONE LLAMA.CPP
# ==========================================
print("Recupero dei pesi quantizzati GGUF (Llama-3.1-8B-Instruct, 4-bit)...")
modello_gguf_path = hf_hub_download(
    repo_id="bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
    filename="Meta-Llama-3.1-8B-Instruct-Q8_0.gguf"
)

print("Inizializzazione del motore di inferenza MPS (Metal)...")
llm = Llama(
    model_path=modello_gguf_path,
    n_gpu_layers=-1,
    n_ctx=4096,
    verbose=False
)

# ==========================================
# 4. GESTIONE DEI PROMPT SEMANTICI
# ==========================================
# Regola 5 eliminata come richiesto
# ============================================================
# SISTEMA PROMPT (più breve — gli esempi fanno il lavoro pesante)
# ============================================================
prompt_sistema  ='''
You are an expert data analyst specializing in humanitarian datasets and food security. Your task is to rewrite reports into anonymous, structurally uniform summaries optimized strictly for vector embedding extraction and semantic similarity analysis.

MANDATORY RULES:
1. FLUENT ANONYMIZATION: Never use bracketed placeholders like [LOCATION] or [DATE]. Rewrite the text using natural, flowing language.
2. SPATIAL TYPOLOGY: Remove all specific geographical entities, toponyms, and local landmarks. Replace them with precise contextual descriptors (e.g., "a landlocked agricultural region", "a conflict-affected urban center", "coastal municipalities").
3. TEMPORAL ABSTRACTION: Eliminate exact years, months, and dates. Standardize temporal references to relative durations (e.g., "a six-month period") and seasonal or agricultural cycles (e.g., "the lean season", "post-harvest period", "monsoon season").
4. ENTITY & PROXY ABLATION: Strip out specific names of local factions, armed groups, ethnic minorities, or localized NGOs that serve as direct geographic identifiers.
5. CAUSAL & TECHNICAL PRESERVATION: Meticulously retain all mechanical drivers of food insecurity (e.g., localized inflation, currency depreciation, supply chain disruption, drought) and all technical metrics (e.g., IPC classifications, percentage changes, kilocalorie deficits).
6. OUTPUT FORMAT: Output strictly the rewritten text as a single cohesive paragraph. Do not include introductory filler, concluding remarks, bullet points, or formatting tags.'''

# ============================================================
# PROMPT STRUTTURA CON ESEMPI
# ============================================================
prompt_struttura = (
    f"--- NOW PROCESS THIS REPORT ---\n"
    f"INPUT:\n"
)




# ==========================================
# (Inserisci questo blocco prima del ciclo FOR)
# ==========================================
nome_modello = "Llama_3_1_8B_8bit"
colonna_output = f"{nome_modello}_anonimo_promtp_in_context"

# Inizializza la colonna nel DataFrame se non esiste per evitare KeyError
if colonna_output not in df_input.columns:
    df_input[colonna_output] = pd.NA

print(f"\nTrovati {len(df_input)} report nel DataFrame da elaborare...\n")

# ==========================================
# 5. ELABORAZIONE DEL BATCH DAL DATAFRAME
# ==========================================
for index, row in tqdm(df_input.iterrows(), total=len(df_input), desc="Elaborazione report"):

    # Lookup vettoriale per evitare inferenze ridondanti:
    # Se la colonna contiene già un valore per questa riga, salta.
    if pd.notna(row.get(colonna_output)):
        continue

    nome_file = row['nome_file']
    testo_originale = str(row['testo_originale'])
    testo_pulito = re.sub(r'^.*?={20,}\n*', '', testo_originale, flags=re.DOTALL).strip()

    if not testo_pulito or testo_pulito.lower() == 'nan':
        print(f" -> Avviso: il testo per {nome_file} risulta vuoto dopo la pulizia.")
        continue

    messages = [
        {"role": "system", "content": prompt_sistema},
        {"role": "user", "content": f"{prompt_struttura}\n\nReport:\n{testo_pulito}"}
    ]

    try:
        outputs = llm.create_chat_completion(
            messages=messages,
            max_tokens=700,
            temperature=0.0
        )

        scheda_anonima = outputs["choices"][0]["message"]["content"].strip()
        scheda_anonima = re.sub(r'###.*?\n', '', scheda_anonima).strip()

        # Inserimento atomico del risultato nel DataFrame in RAM
        df_input.loc[index, colonna_output] = scheda_anonima

    except Exception as e:
        print(f"\nErrore di esecuzione sul file {nome_file}: {e}")
        time.sleep(2)



Recupero dei pesi quantizzati GGUF (Llama-3.1-8B-Instruct, 4-bit)...


Inizializzazione del motore di inferenza MPS (Metal)...

Trovati 497 report nel DataFrame da elaborare...



Elaborazione report: 100%|██████████| 497/497 [1:27:54<00:00, 10.61s/it]


In [3]:
df_input.to_csv("dataset_report_anonimizzati_llama_3_1_8B_8bit_prompt_gemini.csv")